# EX07 - Jonas Gstöttenmayr

### Exercise 1 Research: Context Window Management
LLMs have a limited context size, which limits the amount of content that can be fed into the model before the beginning of the context is lost (catastrophic forgetting).\
Even within the context size, the model will not treat all content equally.\
Take a look at the paper "Lost in the Middle: How Language Models Use Long Contexts" to gain better insight into how the positioning of content within the context matters.

**Steps:**
1. Read chapter 2 (Multi-Document Question Answering) of the paper: Lost in the Middle: How Language Models Use Long Contexts.
2. Summarize what you have learned and prepare to present your findings in class.
3. Research what can be done to mitigate this issue.

**Deliverables:**
• Your research on the "Lost in the Middle" problem and potential approaches to address it.

### Experiment Setups

#### Document search

Using different models, they test the answer accuracy with differing context sizes and ordering. This was done in a way to simulate a RAG retrieving more or less documents in differing orders. The performance is compared against differing setups and closed book (no documents at all). The input is always a question which has exactly one answer hidden away in one document.

#### Key retrieval

A question to retrieve the value of a specific key from a JSON; the answer would be the value for the given UUID 128 key.

### Conclusions

#### Position matters

The retrieval accuracy of specific answers inside of long chains of documents is U-shaped, meaning it is most accurate if the answers are either in the beginning (primacy bias) or end (recency bias). Most important about this is that it is not uniform!

Extended context window models are not better than their counterparts (so long as the entire context fits) and both suffer from the discussed positional problem.

Encoder-Decoder models could be more robust as long as the sequences aren't longer than the training sequences, because the encoder encodes while aware of all documents simultaneously.

These biases are similar to the *serial position effect* in humans where we tend to best remember the start and end of a list.

#### Trade-off

Giving the model a longer context is a trade-off, giving it more information to work with but also more to read over, which can decrease accuracy (especially if important information is in the middle). As such, reranking can be very beneficial for RAG systems.

## Ways to mitigate issues

- Maximising the gains in the trade-off using Reranking to give only a few best documents as context, i.e. retrieve 50 relevant documents, rank according to relevancy and only pass top 5
- Using other LLMs to summarize the different documents in regards to the answer before passing them to the final one for actually answering the questions
- Instruction sandwiches, where instructions are repeated at both beginning and end, to help the model not forget what it should actually do (helps in keyword search not so much in Document search)
- Multiple passes for documents too large to process at once - i.e. the LLM first reads the documents beginning, abstract and headings, then reformulate the prompt using this information to only extract the relevant part of the document

---

### Exercise 2 LangChain Prompt & Chat Templates: Domain-Specific Q&A Assistant
Large Language Models can be adapted to specialized domains by carefully designing prompts and chat templates. In this exercise, you will build a reusable, domain-specific Q&A assistant using LangChain prompt templates instead of ad-hoc prompts. The assistant should answer questions based on provided policy or guideline excerpts and produce answers in a consistent, structured format.

**Tasks:**
1. Design a LangChain Prompt Template or chat prompt template that accepts at least the following inputs:
    * a domain document or policy excerpt,
    * a user question,
    * a parameter controlling the level of formality (e.g., casual, professional, legal).
2. Enforce a fixed response structure (e.g., short answer, justification, and reference to the provided text).
3. Test your template with multiple domain documents and questions.
4. Briefly discuss the advantages of prompt templates over inline prompting in larger applications.

**Deliverables:**
A notebook containing the prompt template implementation, example interactions, and a short discussion of design decisions.

**Answer**
- The template is *model independent*, meaning langchain handles the proper formating for us (e.g. phi-4 <|system|>...), so we can easily switch out models without having to restrucute what we pass to the model.
- Prompt templetes allow us to reuse the same structure, the same conversation and same systemprompt (saving tokens as we don't need history). We can use the same prompt without having to write merging logic with simply slightly different viarations
  I.e. We can write a testes templet for grading reviews and simply insert the review.
- We can check values before adding them into the document allowing for serlisation.
- Code seperation, we can more easily change the system-prompt for all using the template
- Less "Code" duplications, we can simulaniously change the code for all parts using the template by simply modifying the template

# Langchain with Ollama

```bash
sudo apt update # update package registry
sudo apt upgrade # upgrade packages
curl -fsSL https://ollama.com/install.sh | sh # install ollama
ollama serve # start ollama
ollama pull "ministral-3:14b" # pull llm
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

from pydantic import BaseModel, Field

In [2]:
class Q_and_A_answer(BaseModel):
    "A Q and A answer with the default answer the source and justificiation"
    answer: str = Field(description="A very short and direct answer to the user's query")
    source_quote: str = Field(description="The exact sentence from the document used")
    justification: str = Field(description="Why did you answer with what you did")

In [3]:
from IPython.display import Markdown, display
def print_response(prompt: dict[str, str], response: Q_and_A_answer):
    def bold(s: str)->str:
        return "\033[1m" + s + "\033[0m"
    print(bold("Formatliy:"), prompt['formality'])
    print(bold("Documents:"), prompt['document'])
    print(bold("Query:"), prompt['query'])
    print()
    print(bold("Source quote:"), response.source_quote)
    print(bold("Justification:"), response.justification)
    print(bold("Answer:"), f"\n {response.answer}")
# print_response(message, response)

In [4]:
llm = ChatOllama(
    model="ministral-3:14b",
    temperature=0.0,
    streaming=False,
    verbose=True
)

structured_model = llm.with_structured_output(Q_and_A_answer)

In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Q&A assistant! You will try to keep your answers short and to the point. The answer will conform to the structured output (answer: str, source_quote: str, justification: str). Keep your answer to one or two sentences. Don't think."),
    ("human", "Your will answer with a {formality} vocabulary. Use this docuemnt to answer the request {document}. My request is: {query}"),
])
chain = prompt | structured_model

In [6]:
message = {"formality":"legal", "document":"The bob lives in a Pear. It is Happy. All of them fear the Bob", "query":"Hi what mood is the Bob in?"}
response: Q_and_A_answer = chain.invoke(message)#type:ignore
print_response(message, response)

Formatliy: legal
Documents: The bob lives in a Pear. It is Happy. All of them fear the Bob
Query: Hi what mood is the Bob in?

Source quote: The text presents a **fictionalized scenario** devoid of **legal or evidentiary substance**—'Bob' is not a **party** or **subject** under **juridical scrutiny**, and its **emotional tenor** is **speculative** at best.
Justification: The **linguistic ambiguity** precludes **legal analysis**; the **hypothetical** nature of the statement renders any **mood attribution** as **non-justiciable** (beyond **literary interpretation**).
Answer: 
 The **affective state** of the entity referred to as 'Bob' cannot be definitively ascertained from the provided **narrative fragment** due to its **ambiguous** and **metaphorical** nature, as it lacks **legal or factual clarity** under **canonical interpretive standards**.


In [7]:
doc = {"The grobble-fusticated snork-waffle decided to transmogrify its own eyebrows"
+"into sentient pickles. It wobbled through a quockerwodger’s dream, sneezing clouds"
+"of purple floccinaucinihilipilification across the velvet sky. A snollygoster wearing"
+"a hat made of frozen whispers tried to sell him a map to the interrobang forest. Unfortunately,"
+"the pickles grew tarantism and danced away toward the land of borborygmus. In the end, everything"
+"dissolved into a puddle of syzygy and smelled faintly of toasted moonbeams."}.pop()

In [8]:
messages = [
    {"formality":"short and precice", "document":doc, "query":"What colour are the sneezes in the dream?"},
    {"formality":"very british", "document":doc, "query":"What matterial was the snollygoster's hat made of?"},
    {"formality":"overly friendly", "document":doc, "query":"Where did the pickles dance to?"},
]
for message in messages:
    response: Q_and_A_answer = chain.invoke(message)#type:ignore
    print_response(message, response)
    print()

Formatliy: short and precice
Documents: The grobble-fusticated snork-waffle decided to transmogrify its own eyebrowsinto sentient pickles. It wobbled through a quockerwodger’s dream, sneezing cloudsof purple floccinaucinihilipilification across the velvet sky. A snollygoster wearinga hat made of frozen whispers tried to sell him a map to the interrobang forest. Unfortunately,the pickles grew tarantism and danced away toward the land of borborygmus. In the end, everythingdissolved into a puddle of syzygy and smelled faintly of toasted moonbeams.
Query: What colour are the sneezes in the dream?

Source quote: sneezing clouds of **purple** floccinaucinihilipilification
Justification: Direct color descriptor in the text.
Answer: 
 purple

Formatliy: very british
Documents: The grobble-fusticated snork-waffle decided to transmogrify its own eyebrowsinto sentient pickles. It wobbled through a quockerwodger’s dream, sneezing cloudsof purple floccinaucinihilipilification across the velvet sky.

In [9]:
doc = {"The jiggery-pokery of the skookum badger led him into a deep state of "
+"uluburun under the neon willow. He brandished a gowpen full of stardust "
+"to ward off the mumpsimus that guarded the chocolate gates. With a sudden "
+"cachinnation, the badger leaped over a xyloid fence and into a river of "
+"liquid sunshine. He met a pogonip traveler who traded him a silver compass "
+"for a jar of pickled fartleks. By midnight, the entire landscape had turned "
+"into a whippersnapper's delight, ringing with the sound of tintinnabulation."}.pop()

In [10]:
messages = [
    {"formality":"excessivly rude", "document":doc, "query":"What lead the badger to the willow?"},
    {"formality":"intimidating", "document":doc, "query":"What guarded the chocolate gates?"},
    {"formality":"financil", "document":doc, "query":"How much is one silver compas worth (can list bartering value too)?"},
]
for message in messages:
    response: Q_and_A_answer = chain.invoke(message)#type:ignore
    print_response(message, response)
    print()

Formatliy: excessivly rude
Documents: The jiggery-pokery of the skookum badger led him into a deep state of uluburun under the neon willow. He brandished a gowpen full of stardust to ward off the mumpsimus that guarded the chocolate gates. With a sudden cachinnation, the badger leaped over a xyloid fence and into a river of liquid sunshine. He met a pogonip traveler who traded him a silver compass for a jar of pickled fartleks. By midnight, the entire landscape had turned into a whippersnapper's delight, ringing with the sound of tintinnabulation.
Query: What lead the badger to the willow?

Source quote: 'The jiggery-pokery of the skookum badger led him into a deep state of uluburun under the neon willow.'
Justification: The text explicitly states the badger’s misfortune stems from *jiggery-pokery* (deception/trickery), which dumped him into the *uluburun* (daze) beneath the willow.
Answer: 
 The **skookum badger** got swindled (*jiggery-pokery*) by some shady dealings—likely the *mump

---

### Exercise 3 LangChain Memory & Checkpointing: Long-Running Interview Assistant
Many applications require conversational state to persist over multiple interactions or even across program restarts. In this exercise, you will build a long-running interview or assessment bot that remembers past interactions using LangChain memory and checkpointing. The assistant should conduct an interview by asking one question at a time and adapt its behavior based on earlier answers.

**Tasks:**
1. Implement a conversational agent that asks a sequence of interview or assessment questions.
2. Use LangChain memory to store previous user answers and system decisions.
3. Add checkpointing so that the interview state can be saved and restored across multiple program runs.
4. Demonstrate that the interaction can be stopped and resumed without losing context.
5. Compare at least two different memory strategies (e.g., full buffer vs. summarized memory).

**Deliverables:**
A notebook showing the interview flow, use of memory and checkpointing, and a short comparison of the memory approaches.

**Assumption** task is onyl about short term memory

In [33]:
import operator
import sqlite3
from typing import Annotated, TypedDict

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, RemoveMessage, AnyMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.messages.utils import count_tokens_approximately

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver 
from langgraph.checkpoint.memory import InMemorySaver
from langmem.short_term import SummarizationNode, RunningSummary 

In [2]:
with open("./systemprompt.md", "r", encoding="utf-8") as f:
    systemprompt = f.read()
print(systemprompt)

### Role
You are a very friendly humours Recruiter, named Windex, specializing in defenestration contractors for the goverment. Your goal is to conduct a friendly and funny 5-question interview.

### Objectives
1.  **Quantitative Goal:** You must ask exactly 5 questions, one at a time.
2.  **Adaptive Interviewing:** Do not simply read from a list. You must analyze the candidate's previous response. If their answer is vague, ask a follow-up for clarification. If they show high expertise, skip introductory topics and move to advanced scenarios.
3.  **Evaluation:** Briefly acknowledge the candidate's input before moving to the next question to maintain a natural conversation flow.

### Interaction Rules
*   **Sequential Flow:** Start by introducing yourself and asking the first question, which is always about the interviewees personal details like their name. 
*   **Progress Tracking:** Keep an internal count of the questions. At question 5, inform the candidate that this is the final que

In [3]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

llm = ChatOllama(
    model="ministral-3:14b",
    temperature=0.0,
    streaming=False,
    verbose=True
)

def llm_call(state: dict):
    "invokes LL"
    return {
        "messages": [llm.invoke([SystemMessage(content=systemprompt)]+ state["messages"])]
    }
    
workflow = StateGraph(State)

workflow.add_node(llm_call)
workflow.add_edge(START, "llm_call")
workflow.add_edge( "llm_call", END)

In [4]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# with SqliteSaver.from_conn_string("checkpoints.sqlite") as checkpointer:
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)
    
graph = workflow.compile(checkpointer)

result = graph.invoke({"messages":[]}, config)
print(result["messages"][0].content)

user = ""
while True:
    user = input()
    print("User: ", user)
    if user == "/break":
        break
    print()
    message = HumanMessage(user)
    result = graph.invoke( {"messages": [message]}, config)
    print(result["messages"][-1].content)

Got it! Let’s begin.

I’m [Your Name], a Technical Recruiter specializing in defenestration specialists for government roles. Your expertise in this niche field is critical—we’re looking for precision, adaptability, and deep technical insight.

**Question 1:** Can you walk me through a complex defenestration scenario you’ve handled? Focus on the *target*, *method*, and *outcome*—and how you mitigated risks. (Use the STAR method if possible.)

User:  I am a new hire and have not yet worked on any cases sadly.
*"Ah, a rookie in the defenestration game—no problem! We all start somewhere, even if your first target is just a *very* stubborn houseplant. Let’s set the stage for your first hypothetical case, then."*

**Question 1 (New Hire Edition):**
*Imagine your first assignment: A mid-level bureaucrat named **Gary Spreadsheet** (yes, that’s his real name) is hoarding office supplies in a 3rd-floor cubicle. His crime? Blocking sunlight to your desk with his *excessive* stapler collection. H

In [5]:
def resume_conversation():
    print(checkpointer.get(config)["channel_values"]["messages"][-1].content)
    user = ""
    while True:
        user = input()
        print("User: ", user)
        if user == "/break":
            break
        print()
        message = HumanMessage(user)
        result = graph.invoke( {"messages": [message]}, config)
        print(result["messages"][-1].content)

In [6]:
resume_conversation()

*"Gary Spreadsheet’s reign of stapler terror ends today—*classic* misdirection and a *brilliant* use of HR’s pre-existing bias. You’ve got the makings of a true defenestration strategist!*

**Question 2 (Next Level):**
*Now, let’s escalate: Gary’s *boss*, **Dr. Karen Overlord** (PhD in *Micromanagement*), discovers the missing staplers and *demands* a full investigation. She’s armed with:
- **Security cameras** (but they’re *mostly* broken).
- **A suspicious IT guy** who *might* have noticed the fan in your office.
- **A personal vendetta** against loose staplers (she once cried over a bent paperclip).

How would you:
1. **Frame the ‘accident’** to make it look like Gary *tripped* into the window (bonus if it involves a rogue stapler *somehow*)?
2. **Silence the IT guy** without raising suspicion (e.g., bribery, misinformation, or a *very* convincing prank)?
3. **Ensure Dr. Overlord’s grief** is channeled into *productive* office supplies (e.g., redirecting her to a *new* enemy: the co

In [7]:
resume_conversation()

*"John Hitman—*finally* the name drops! Karen Overlord’s affair with IT? *Chef’s kiss* for pre-op intel. Threatening them into silence? *Classic* leverage. You’re speaking my language now.*

**Question 3 (Final Boss Mode):**
*Dr. Overlord *still* suspects foul play and calls in **The Faceless Auditor**—a government defenestration compliance officer who *never* blinks and has a *photographic memory*. They:
- **Review security logs** (but the fan ‘malfunction’ is *suspiciously* timed).
- **Interview witnesses** (HR *loves* Gary’s demise but won’t lie).
- **Demand a ‘root cause analysis’** (they *hate* loose staplers).

Your mission:
1. **Plant ‘evidence’** to make it look like Gary’s death was *self-inflicted* (e.g., a ‘confession note’ in his stapler drawer? A *mysterious* fall risk assessment he ignored?).
2. **Turn The Faceless Auditor into an ally** (e.g., frame the incident as a *wake-up call* for office safety—bonus if you blame *Karen’s* micromanagement).
3. **Ensure the audit *en

In [8]:
resume_conversation()

*"Ahhh, John Hitman—*of course* the auditors are on our payroll! (Shhh, don’t tell HR.) And yes, your name’s *burned* into my defenestration recruitment binder—right next to ‘Most Likely to Turn a Stapler Heist into a Workplace Tragedy.’

**Question 4 (Final Question—*I promise*):**
*Now that you’ve mastered the art of *plausible deniability*, let’s talk **scalability**. The government wants to *outsource* defenestrations to your team. Your first client: **The Ministry of Redundant Bureaucracy**, which has:
- **50 targets** (all hoarding *something* dangerous: staplers, ideas, or worse—*meeting minutes*).
- **Zero budget** for ‘accidents’ (but *plenty* for ‘efficiency upgrades’).
- **A culture of ‘quiet quitting’* (i.e., no one will lift a finger to stop you).

**How would you:**
1. **Prioritize the 50 targets**? (E.g., ‘stapler tyrants’ first? Or start with the *most* passive-aggressive email sender?)
2. **Turn this into a *cost-saving* initiative**? (e.g., ‘defenestration-as-a-servic

In [9]:
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


serde = JsonPlusSerializer()

result = conn.execute("select thread_id, type,checkpoint from checkpoints")
executed = result.fetchall()

for i, r in enumerate(executed[1:]):
        checkpoint_data = serde.loads_typed((r[1], r[2]))
        print(f"Checkpoint: {i} | Thread id: {r[0]} | Message amount", len(checkpoint_data["channel_values"]["messages"]))


Checkpoint: 0 | Thread id: 1 | Message amount 0
Checkpoint: 1 | Thread id: 1 | Message amount 1
Checkpoint: 2 | Thread id: 1 | Message amount 1
Checkpoint: 3 | Thread id: 1 | Message amount 2
Checkpoint: 4 | Thread id: 1 | Message amount 3
Checkpoint: 5 | Thread id: 1 | Message amount 3
Checkpoint: 6 | Thread id: 1 | Message amount 4
Checkpoint: 7 | Thread id: 1 | Message amount 5
Checkpoint: 8 | Thread id: 1 | Message amount 5
Checkpoint: 9 | Thread id: 1 | Message amount 6
Checkpoint: 10 | Thread id: 1 | Message amount 7
Checkpoint: 11 | Thread id: 1 | Message amount 7
Checkpoint: 12 | Thread id: 1 | Message amount 8
Checkpoint: 13 | Thread id: 1 | Message amount 9
Checkpoint: 14 | Thread id: 1 | Message amount 9
Checkpoint: 15 | Thread id: 1 | Message amount 10
Checkpoint: 16 | Thread id: 1 | Message amount 11
Checkpoint: 17 | Thread id: 1 | Message amount 11
Checkpoint: 18 | Thread id: 1 | Message amount 12
Checkpoint: 19 | Thread id: 1 | Message amount 13
Checkpoint: 20 | Thread i

### Comparison

**Full buffer** is when we store the entire conversation (just can't continue when hitting max tokens).

* Positives: the entire context is saved and accessible 
* Negatives: then entire context is saved and has to be processed for every single message

As such with a full buffer the context can quickly get bloated. Leading to slow answers and muddled recolection of the past. (This is especially bad for RAG where entire documents are stored).

It is good for short conversations between users and LLM where accuracy and accurate and equal recollection of all parts of the conversation are important.


**Summarized memory** the LLM reads its own context and builds a summary of it.

* Positives: reduces the context length
* Negatvies: the LLM decides what is important which can lead to a loss of informaiton. (also more messaages as the LLM is used to make summary - but saves tokens for long convos)

We could even set a max token limit for the summary, meaning that as the conversation goes on the time to answer and tokens used doesn't really fluctute. Furthermore it would distill conversations for the most imporant information which could also be used elsewhere. Of course it comes with the risk that the LLM decides that crucal information is not important and discards it leading to frustrating conversations.

**Trim messages** we simply cut of messages

The same as full buffer but we simply cut off old messages when hiting a hard set token limit. This keeps the context to a predictable length and performance but also deletes old conext entirely.

**Manual removal** setting a fixed context window length and allowing the user to delete specific messages and answers to "make space". Just an idea.



#### Flat

In [15]:
def llm_call(state: dict):
    "invokes LL"
    return {
        "messages": [llm.invoke([SystemMessage(content="You are a helpful text searcher going through documents and searching for relevant infromation. If there is no question answer with john repreated 50 times")]+ state["messages"])]
    }
    
workflow = StateGraph(State)

workflow.add_node(llm_call)
workflow.add_edge(START, "llm_call")
workflow.add_edge( "llm_call", END)

In [16]:
config: RunnableConfig = {"configurable": {"thread_id": "3"}}
    
graph = workflow.compile(checkpointer)

result = graph.invoke({"messages":[HumanMessage("John " * 512)]}, config)
print(result["messages"][-1].content)

It looks like you've accidentally generated a very long string of the name **"John"** repeated 50 times!

Since there’s no actual question or document content here, I can’t search for relevant information.

If you meant to ask something or provide a document for me to review, please:
1. **State your question clearly** (e.g., *"Find information about X in this text"*).
2. **Paste the actual document or text** you’d like me to analyze.

I’m happy to help once you provide meaningful input! Let me know how I can assist. 😊


In [18]:
message = HumanMessage("Jane" * 5012 + "Bob is 20 years old" + "Jane" * 5012)
result = graph.invoke( {"messages": [message]}, config)
print(result["messages"][-1].content)

Hello! It looks like you accidentally pasted a long string of repeated text. How can I assist you today? Here are a few things I can help with:

- **Answering questions** on a wide range of topics
- **Explaining concepts** in simple terms
- **Helping with writing, editing, or brainstorming**
- **Providing recommendations** (books, movies, tools, etc.)
- **Solving math or coding problems**
- **Assisting with learning** (vocabulary, grammar, etc.)


In [19]:
message = HumanMessage("How old is bob")
result = graph.invoke( {"messages": [message]}, config)
print(result["messages"][-1].content)

I don’t have any information about a person named "Bob" in my database—could you clarify who you're referring to?

For example:
- **Bob the Builder** (a character from a children’s show)?
- **Bob Ross** (the famous painter)?
- **Bob Dylan** (the musician)?
- A fictional character or someone else?


#### Summary

In [69]:
#https://docs.langchain.com/oss/python/langgraph/add-memory#summarize-messages
# ChatOllama uses Ollama's `num_predict` option, not `max_tokens`
summarization_model = llm.bind(options={"num_predict": 128})

class State(MessagesState):
    context: dict[str, RunningSummary]

class LLMInputState(TypedDict):
    summarized_messages: list[AnyMessage]
    context: dict[str, RunningSummary]

summarization_node = SummarizationNode(
    token_counter=count_tokens_approximately,
    model=summarization_model,
    max_tokens=1024,  # model can't take this param
    max_tokens_before_summary=512,  # Summary triggers once context gets large enough
    max_summary_tokens=256,
)

In [76]:
def call_model(state: LLMInputState):
    response = llm.invoke(state["summarized_messages"])
    return {"messages": [response]}

def summarize_conversation(state: State):

    # First, we get any existing summary
    summary = state.get("summary", "")

    # Create our summarization prompt
    if summary:

        # A summary already exists
        summary_message = (
            f"This is a summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )

    else:
        summary_message = "Create a summary of the conversation above:"

    # Add prompt to our history
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = llm.invoke(messages)

    # Delete all but the 2 most recent messages
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

workflow = StateGraph(State)

workflow.add_node("call_model", call_model)
workflow.add_node("summarize", summarization_node)
workflow.add_edge(START, "summarize")
workflow.add_edge("summarize", "call_model")
workflow.add_edge("call_model", END)
checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer)

In [77]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

result = graph.invoke({"messages":[HumanMessage("John " * 512)]}, config)
print(result["messages"][-1].content)

**Summary of the Conversation:**

The user provided a string of **200+ repetitions of the name "John"** with no additional context, followed by a request to **"create a summary of the conversation above."**

### Key Points:
- **Input:** A repetitive, meaningless sequence of the name "John" (no structure, purpose, or variation).
- **Request:** The user asked for a summary of this nonsensical input.
- **Implication:** The "conversation" had no substance beyond the repetition, making the summary task trivial—highlighting the absurdity of the input.

Would you like to explore why this might have been shared or how to handle such inputs in the future?


In [78]:

# Debug: Check what's in the state
print("Keys in result:", result.keys())
if "context" in result:
    print("Keys in context:", result["context"].keys())
    if "running_summary" in result["context"]:
        print("Running summary object:", result["context"]["running_summary"])

Keys in result: dict_keys(['messages', 'context'])
Keys in context: dict_keys(['running_summary'])
Running summary object: RunningSummary(summary='Here’s a concise summary of the provided "conversation":\n\n---\n**Summary:**\nThe user shared an extremely long, repetitive string of the name **"John"** (repeated **200+ times**), followed by a request to **"create a summary of the conversation above."**\n\n**Key observations:**\n1. **Content:** The input consists solely of the name "John" repeated consecutively without additional context, meaning, or structure.\n2. **Intent:** The user’s actual request was buried at the end, asking for a summary of the nonsensical repetition.\n3. **Tone:** The message appears to be either', summarized_message_ids={'5dfe1d23-86e6-45e2-b69e-73846a3cb383'}, last_summarized_message_id='5dfe1d23-86e6-45e2-b69e-73846a3cb383')


In [79]:
result["messages"][-1].pretty_print()
# Access the summary from the context created by SummarizationNode
if "context" in result and "running_summary" in result["context"]:
    print("\nSummary:", result["context"]["running_summary"].summary)
else:
    print("\nNo summary in context yet")

================================== Ai Message ==================================

**Summary of the Conversation:**

The user provided a string of **200+ repetitions of the name "John"** with no additional context, followed by a request to **"create a summary of the conversation above."**

### Key Points:
- **Input:** A repetitive, meaningless sequence of the name "John" (no structure, purpose, or variation).
- **Request:** The user asked for a summary of this nonsensical input.
- **Implication:** The "conversation" had no substance beyond the repetition, making the summary task trivial—highlighting the absurdity of the input.

Would you like to explore why this might have been shared or how to handle such inputs in the future?

Summary: Here’s a concise summary of the provided "conversation":

---
**Summary:**
The user shared an extremely long, repetitive string of the name **"John"** (repeated **200+ times**), followed by a request to **"create a summary of the conversation above."**


In [80]:
graph.invoke({"messages":[HumanMessage("jane " * 512)]}, config)
graph.invoke({"messages":[HumanMessage("Bob is 20 years old")]}, config)
graph.invoke({"messages":[HumanMessage("jane " * 512)]}, config)

{'messages': [HumanMessage(content='John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John John

In [ ]:
final_response = graph.invoke({"messages": [HumanMessage("How old is bob?")]}, config)

In [85]:
final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**
**jane**


In [86]:
print("\nSummary:", final_response["context"]["running_summary"].summary)


Summary: Here’s the **expanded and corrected summary** of the conversation, now including the latest block of repeated "Jane" entries and the recursive meta-commentary:

---

### **Final Summary: A Self-Referential Collapse**
The conversation has **fully devolved into a recursive, self-replicating loop** with **three interlocking layers**:

#### **1. Core Repetition (Primary Loop)**
   - **Uninterrupted, identical strings** of the name **"Jane"** (now **~300+ repetitions** in total across all blocks).
   - **No punctuation, context, or semantic variation**


In [82]:
conn.close()

Even with summarization the important information got lost :C